# Yelp Data Loading and EDA\n
\n
This notebook starts the local-first data inspection workflow for `yelp-cost-aware-agent`.\n
\n
Goals for this first pass:\n
- inspect the archive contents without fully extracting everything first\n
- load manageable samples of businesses and reviews\n
- understand dataset shape, schema, and obvious quality issues\n
- identify features that matter for cost-aware retrieval and ranking\n
\n
Assumptions:\n
- the Yelp archive lives at `../Yelp JSON/yelp_dataset.tar`\n
- the archive contains JSON or JSONL business/review files\n
- we want sample-based EDA first, not a full heavy ingestion pass

In [3]:
from __future__ import annotations
import json
import tarfile
from itertools import islice
from pathlib import Path

import pandas as pd

ARCHIVE_PATH = Path('../Yelp JSON/yelp_dataset.tar')
ARCHIVE_PATH.exists(), ARCHIVE_PATH

(True, PosixPath('../Yelp JSON/yelp_dataset.tar'))

In [4]:
def list_archive_members(archive_path: Path) -> list[str]:
    with tarfile.open(archive_path, 'r:*') as tf:
        return [member.name for member in tf.getmembers()]

members = list_archive_members(ARCHIVE_PATH)
len(members), members[:20]

(6,
 ['Dataset_User_Agreement.pdf',
  'yelp_academic_dataset_business.json',
  'yelp_academic_dataset_checkin.json',
  'yelp_academic_dataset_review.json',
  'yelp_academic_dataset_tip.json',
  'yelp_academic_dataset_user.json'])

In [5]:
def find_member(members: list[str], keyword: str) -> str:
    keyword = keyword.lower()
    matches = [name for name in members if keyword in name.lower()]
    if not matches:
        raise ValueError(f'No archive member found for keyword={keyword!r}')
    return matches[0]


business_member = find_member(members, 'business')
review_member = find_member(members, 'review')
business_member, review_member

('yelp_academic_dataset_business.json', 'yelp_academic_dataset_review.json')

In [8]:
def load_json_lines_from_tar(archive_path: Path, member_name: str, nrows: int | None = None) -> pd.DataFrame:
    rows = []
    with tarfile.open(archive_path, 'r:*') as tf:
        extracted = tf.extractfile(member_name)
        if extracted is None:
            raise FileNotFoundError(member_name)
        iterator = extracted if nrows is None else islice(extracted, nrows)
        for raw_line in iterator:
            if not raw_line:
                continue
            rows.append(json.loads(raw_line))
    return pd.DataFrame(rows)

businesses = load_json_lines_from_tar(ARCHIVE_PATH, business_member, nrows=20000)
reviews = load_json_lines_from_tar(ARCHIVE_PATH, review_member, nrows=50000)

businesses.shape, reviews.shape

((20000, 14), (50000, 9))

## Schema inspection

In [9]:
businesses.head(3)

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,0,{'ByAppointmentOnly': 'True'},"Doctors, Traditional Chinese Medicine, Naturop...",None
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,1,{'BusinessAcceptsCreditCards': 'True'},"Shipping Centers, Local Services, Notaries, Ma...","{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:30', ..."
2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223236,-110.880452,3.5,22,0,"{'BikeParking': 'True', 'BusinessAcceptsCredit...","Department Stores, Shopping, Fashion, Home & G...","{'Monday': '8:0-22:0', 'Tuesday': '8:0-22:0', ..."


In [10]:
reviews.head(3)

,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11
1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5.0,1,0,1,I've taken a lot of spin classes over the year...,2012-01-03 15:28:18
2,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3.0,0,0,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30


In [11]:
businesses.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   business_id   20000 non-null  object 
 1   name          20000 non-null  object 
 2   address       20000 non-null  object 
 3   city          20000 non-null  object 
 4   state         20000 non-null  object 
 5   postal_code   20000 non-null  object 
 6   latitude      20000 non-null  float64
 7   longitude     20000 non-null  float64
 8   stars         20000 non-null  float64
 9   review_count  20000 non-null  int64  
 10  is_open       20000 non-null  int64  
 11  attributes    18198 non-null  object 
 12  categories    19985 non-null  object 
 13  hours         16927 non-null  object 
dtypes: float64(3), int64(2), object(9)
memory usage: 2.1+ MB


In [12]:
reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   review_id    50000 non-null  object 
 1   user_id      50000 non-null  object 
 2   business_id  50000 non-null  object 
 3   stars        50000 non-null  float64
 4   useful       50000 non-null  int64  
 5   funny        50000 non-null  int64  
 6   cool         50000 non-null  int64  
 7   text         50000 non-null  object 
 8   date         50000 non-null  object 
dtypes: float64(1), int64(3), object(5)
memory usage: 3.4+ MB


## Missingness and basic quality checks

In [13]:
business_missing = (businesses.isna().mean().sort_values(ascending=False) * 100).round(2)
review_missing = (reviews.isna().mean().sort_values(ascending=False) * 100).round(2)
business_missing.head(15), review_missing.head(15)

(hours           15.36
 attributes       9.01
 categories       0.08
 business_id      0.00
 name             0.00
 address          0.00
 city             0.00
 state            0.00
 postal_code      0.00
 latitude         0.00
 longitude        0.00
 stars            0.00
 review_count     0.00
 is_open          0.00
 dtype: float64,
 review_id      0.0
 user_id        0.0
 business_id    0.0
 stars          0.0
 useful         0.0
 funny          0.0
 cool           0.0
 text           0.0
 date           0.0
 dtype: float64)

## Business-side EDA\n
\n
This helps us reason about retrieval filters, candidate pruning, and ranking features.

In [14]:
businesses[['stars', 'review_count']].describe()

,stars,review_count
count,20000.000000,20000.000000
mean,3.601750,46.069800
std,0.972995,116.873712
min,1.000000,5.000000
25%,3.000000,8.000000
50%,4.000000,15.000000
75%,4.500000,38.000000
max,5.000000,4554.000000


In [15]:
businesses['city'].value_counts().head(20)

city
Philadelphia        1967
Tampa               1244
Tucson              1231
Indianapolis         979
Nashville            921
New Orleans          832
Reno                 815
Edmonton             687
Saint Louis          649
Santa Barbara        496
Boise                372
Clearwater           301
Metairie             216
Saint Petersburg     215
Sparks               196
Franklin             194
Wilmington           192
St. Louis            180
St. Petersburg       159
Meridian             142
Name: count, dtype: int64

In [16]:
if 'categories' in businesses.columns:
    top_categories = (
        businesses['categories']
        .dropna()
        .str.split(', ')
        .explode()
        .value_counts()
        .head(25)
    )
    display(top_categories)
else:
    print('No categories column found.')

categories
Restaurants                  6953
Food                         3753
Shopping                     3214
Home Services                1907
Beauty & Spas                1876
Health & Medical             1570
Nightlife                    1559
Local Services               1511
Automotive                   1424
Bars                         1400
Event Planning & Services    1356
Sandwiches                   1085
Active Life                  1073
American (Traditional)       1034
Pizza                         960
Coffee & Tea                  863
Fast Food                     859
American (New)                840
Breakfast & Brunch            835
Home & Garden                 775
Hotels & Travel               775
Arts & Entertainment          765
Fashion                       750
Burgers                       729
Auto Repair                   725
Name: count, dtype: int64

In [17]:
if 'attributes' in businesses.columns:
    businesses['has_attributes'] = businesses['attributes'].notna()
    businesses['has_attributes'].value_counts(normalize=True).round(3)
else:
    print('No attributes column found.')

In [18]:
price_col = 'attributes' if 'RestaurantsPriceRange2' not in businesses.columns else 'RestaurantsPriceRange2'
price_col

'attributes'

## Review-side EDA\n
\n
Review volume and review length matter directly for retrieval depth, context size, and model cost.

In [19]:
reviews['text_len_chars'] = reviews['text'].fillna('').str.len()
reviews['text_len_words'] = reviews['text'].fillna('').str.split().str.len()
reviews[['stars', 'useful', 'funny', 'cool', 'text_len_chars', 'text_len_words']].describe()

,stars,useful,funny,cool,text_len_chars,text_len_words
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.00000
mean,3.848000,0.889540,0.250440,0.345060,546.992180,100.84424
std,1.350308,1.864481,0.941455,1.072388,499.267814,92.59301
min,1.000000,0.000000,0.000000,0.000000,3.000000,1.00000
25%,3.000000,0.000000,0.000000,0.000000,225.000000,41.00000
50%,4.000000,0.000000,0.000000,0.000000,393.000000,72.00000
75%,5.000000,1.000000,0.000000,0.000000,693.000000,128.00000
max,5.000000,91.000000,38.000000,49.000000,4997.000000,1006.00000


In [20]:
reviews[['business_id', 'text_len_words']].groupby('business_id').agg(['count', 'mean']).head()

text_len_words            
                                count        mean
business_id                                      
--ZVrH2X2QXBFdCilbirsw              7   46.857143
--_9CAxgfXZmoFdNIRrhHA              1   98.000000
-02xFuruu85XmDn2xiynJw              5   62.200000
-0Ym1Wg3bXd_TDz8JtvOQg              4  170.750000
-1MhPXk1FglglUAmuPLIGg              6   71.500000

## Simple cost-aware heuristics\n
\n
These quick checks help guide early policy choices for retrieval and context reduction.

In [21]:
reviews['approx_tokens'] = (reviews['text_len_chars'] / 4).round().astype(int)
reviews['approx_tokens'].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])

count    50000.000000
mean       136.748680
std        124.816947
min          1.000000
50%         98.000000
75%        173.000000
90%        283.000000
95%        378.000000
99%        620.000000
max       1249.000000
Name: approx_tokens, dtype: float64

In [22]:
review_budget_summary = reviews['approx_tokens'].agg(['mean', 'median', 'max']).round(2)
review_budget_summary

mean       136.75
median      98.00
max       1249.00
Name: approx_tokens, dtype: float64

In [23]:
if {'business_id', 'stars', 'review_count', 'city', 'name'}.issubset(businesses.columns):
    underrated = businesses.query('stars >= 4.5 and review_count <= 50')[['business_id', 'name', 'city', 'stars', 'review_count']].head(20)
    display(underrated)
else:
    print('Business schema does not match expected underrated query fields.')

,business_id,name,city,stars,review_count
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ",Santa Barbara,5.0,7
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,Green Lane,4.5,13
13,jaxMSoInw8Poo3XeMJt8lQ,Adams Dental,Clearwater,5.0,10
16,rBmpy_Y1UbBx8ggHlyb7hA,Arizona Truck Outfitters,Tucson,4.5,10
25,PSo_C1Sfa13JHjzVNW6ziQ,Indian Walk Veterinary Center,Newtown,5.0,15
30,fvWn8oXXwbj2l79cochZyw,Altitude Trampoline Park - Boise,Boise,5.0,30
32,8sshLb4UU7emeUDvtJWnpA,DanceLine,Paoli,4.5,11
38,LcAozWCMLGjwRbokaJAKMg,Edwardsville Children's Museum,Edwardsville,4.5,12
39,fSCNwMtNNQY9QT69Cj9fiA,Sierra Pro Events,Sparks,5.0,7
42,lwItZ1Ck3KtpCgG4CPFmpQ,Stomel Elliot Attorney-At-Law,Cherry Hill,5.0,5


## Next steps\n
\n
Once this sample EDA looks good, the next notebook or script should:\n
1. ingest businesses and reviews into DuckDB\n
2. normalize categories and selected attributes\n
3. build fast city/category filters\n
4. precompute review-length and review-count summaries for cost-aware retrieval policies\n
5. create benchmark slices for Phoenix, Las Vegas, and Tempe